# Encode corpus — tự chạy, tự hồi phục, tự ghép

**Cách dùng:** sửa `SHARDS_THIS_ACCOUNT` ở cell 1 → *Save Version* → **Save & Run All (Commit)** →
tắt máy. Notebook chạy trong container riêng của Kaggle, không phụ thuộc trình duyệt của bạn.

**Hết 12 giờ mà chưa xong?** Bấm *Save & Run All* lần nữa. Notebook đọc trạng thái từ Drive,
bỏ qua mảnh đã xong, và `--resume` mảnh đang dở từ đúng chunk cuối cùng đã ghi.

**Ghép:** không cần notebook riêng. Tài khoản nào chạy xong mà thấy **đủ cả N mảnh trên Drive**
thì tự ghép và đẩy bản cuối lên `Drive:<DRIVE_DIR>/final/`.

---
**Nguồn sự thật là Drive, không phải `/kaggle/working`.** Output của một version Kaggle không tự
có mặt ở version sau, nên mọi trạng thái đều đi qua Drive: xong mảnh nào đẩy ngay mảnh đó.

Quy ước: **`.meta.json` chỉ được ghi khi mảnh chạy XONG.** Nên trên Drive:
`meta` có mặt = mảnh hoàn tất · chỉ có `.npy` + `.progress.json` = mảnh dở, sẽ được resume.

> 🔴 Notebook để **Private** (nó gắn secret Drive của bạn, và dữ liệu là của BTC — Điều 11).
> ⚙️ Settings: **GPU T4 ×2**, **Internet ON**, **Secret `RCLONE_CONF_B64`** đã gắn.

## 1. Cấu hình — chỗ duy nhất đổi giữa các tài khoản

In [ ]:
N_SHARDS   = 8                  # 4 tài khoản × 2 GPU
SHARDS_THIS_ACCOUNT = [0, 1]    # tk1 [0,1] · tk2 [2,3] · tk3 [4,5] · tk4 [6,7]
TAKE_LEFTOVERS = True           # xong phần mình thì gánh nốt mảnh tài khoản khác bỏ dở

DEADLINE_HOURS = 11.0           # dừng có trật tự trước mốc 12h của Kaggle
BATCH_SIZE     = 32             # OOM thì 16 hoặc 8
DO_MERGE       = True           # đủ N mảnh thì tự ghép
FORCE_MERGE_ONLY = False        # True = bỏ qua encode, chỉ ghép
SKIP_SMOKE     = False          # True = bỏ bước thử 2.000 chunk

GIT_URL    = 'https://github.com/<org>/<repo>.git'
GIT_BRANCH = 'p3/pipeline-e2e'
CONFIG     = 'configs/v0.4_dense.yaml'
DATASET    = '/kaggle/input/<ten-dataset>/chunks.jsonl'

RCLONE_SECRET = 'RCLONE_CONF_B64'
DRIVE_REMOTE  = 'gdrive'
DRIVE_DIR     = 'DSC2026/embeddings'

## 2. Code + dữ liệu

`assert` đường dẫn dataset ngay tại đây: symlink trỏ vào chỗ không tồn tại vẫn tạo được,
và lỗi sẽ chỉ nổ sau khi đã nạp model — tức sau khi đã tiêu GPU.

In [ ]:
import os, sys, json, time, signal, subprocess

T0 = time.time()
DEADLINE = T0 + DEADLINE_HOURS * 3600
WORK = '/kaggle/working/repo'

if not os.path.exists(WORK):
    subprocess.run(['git','clone','-b',GIT_BRANCH,'--depth','1',GIT_URL,WORK], check=True)
os.chdir(WORK)
print('commit:', subprocess.run(['git','log','--oneline','-1'],capture_output=True,text=True).stdout.strip())

os.makedirs(f'{WORK}/data', exist_ok=True)
assert os.path.exists(DATASET), (
    f'❌ Không thấy {DATASET}. Kiểm tra đã Add Data dataset chứa chunks.jsonl chưa, '
    f'và tên thư mục trong /kaggle/input có đúng không.')
dst = f'{WORK}/data/chunks.jsonl'
if not os.path.exists(dst):
    os.symlink(DATASET, dst)

import torch, transformers
N_GPU = torch.cuda.device_count()
print(f'torch {torch.__version__} · transformers {transformers.__version__} · GPU {N_GPU}')
print(f'hạn tự dừng: {DEADLINE_HOURS}h (Kaggle cắt ở 12h)')

## 3. rclone từ Secret

Chuẩn bị một lần trên Mac: `rclone config` (remote `gdrive`) rồi
`base64 -i ~/.config/rclone/rclone.conf | pbcopy`, dán vào Secret của **từng** tài khoản.

In [ ]:
import base64
from kaggle_secrets import UserSecretsClient

if subprocess.run(['which','rclone'],capture_output=True).returncode:
    subprocess.run('curl -fsSL https://rclone.org/install.sh | bash', shell=True, check=True)

os.makedirs('/root/.config/rclone', exist_ok=True)
with open('/root/.config/rclone/rclone.conf','wb') as f:
    f.write(base64.b64decode(UserSecretsClient().get_secret(RCLONE_SECRET)))
os.chmod('/root/.config/rclone/rclone.conf', 0o600)

DRIVE = f'{DRIVE_REMOTE}:{DRIVE_DIR}'

def rc(*args, check=False):
    r = subprocess.run(['rclone', *args], capture_output=True, text=True)
    if check and r.returncode:
        raise RuntimeError(f'rclone {args[0]} lỗi: {r.stderr[-500:]}')
    return r

assert rc('lsd', f'{DRIVE_REMOTE}:').returncode == 0, '❌ rclone không kết nối được Drive'
rc('mkdir', DRIVE)
print('rclone OK ·', DRIVE)     # cố ý KHÔNG in nội dung config

## 4. Đọc trạng thái từ Drive

Quyết định làm gì dựa trên thứ đang có trên Drive, không dựa vào trí nhớ của session trước.

In [ ]:
def drive_state():
    """→ (mảnh đã xong, mảnh đang dở). meta có mặt = xong; chỉ progress = dở."""
    r = rc('lsjson', DRIVE)
    names = {f['Name'] for f in json.loads(r.stdout or '[]')} if r.returncode == 0 else set()
    done, partial = set(), set()
    for k in range(N_SHARDS):
        stem = f'embeddings.shard{k}of{N_SHARDS}'
        if f'{stem}.meta.json' in names:
            done.add(k)
        elif f'{stem}.progress.json' in names and f'{stem}.npy' in names:
            partial.add(k)
    return done, partial

done, partial = drive_state()
mine = [k for k in SHARDS_THIS_ACCOUNT if k not in done]
if TAKE_LEFTOVERS:
    mine += [k for k in range(N_SHARDS) if k not in done and k not in mine]

print(f'đã xong trên Drive : {sorted(done)}')
print(f'đang dở (sẽ resume): {sorted(partial)}')
print(f'phiên này sẽ chạy  : {mine}')

# Kéo về phần dở của đúng những mảnh mình sắp chạy (mỗi mảnh ~270 MB, đừng kéo thừa)
for k in [k for k in mine if k in partial]:
    stem = f'embeddings.shard{k}of{N_SHARDS}'
    rc('copy', DRIVE, f'{WORK}/data', f'--include={stem}.npy', f'--include={stem}.progress.json', '-P')
    print(f'  ↻ kéo về phần dở của mảnh {k}')

## 5. Thử 2.000 chunk (chỉ lần đầu)

In [ ]:
if not SKIP_SMOKE and not FORCE_MERGE_ONLY and not done and mine:
    subprocess.run([sys.executable,'-u','scripts/encode_corpus.py','--config',CONFIG,'--dry-run'],check=True)
    subprocess.run([sys.executable,'-u','scripts/encode_corpus.py','--config',CONFIG,
                    '--limit','2000','--device','cuda','--batch-size',str(BATCH_SIZE)],check=True)
else:
    print('bỏ qua bước thử')

## 6. Encode — hàng đợi, 2 GPU, tự dừng trước hạn

Mỗi GPU chạy một mảnh; xong mảnh nào **đẩy ngay lên Drive** rồi nhận mảnh kế tiếp.
Chạm hạn thì gửi `SIGINT` cho tiến trình đang chạy — `encode_corpus.py` bắt tín hiệu đó,
flush memmap, ghi sổ tiến độ rồi thoát; phần dở cũng được đẩy lên Drive để lần sau resume.

In [ ]:
def push(k, partial_ok=False):
    stem = f'embeddings.shard{k}of{N_SHARDS}'
    inc = [f'--include={stem}.npy', f'--include={stem}.progress.json']
    if not partial_ok:
        inc.append(f'--include={stem}.meta.json')
    rc('copy', f'{WORK}/data', DRIVE, '-P', '--stats=60s', *inc, check=True)
    ok = rc('check', f'{WORK}/data', DRIVE, '--one-way', *inc).returncode == 0
    if not partial_ok:
        # Mảnh đã xong thì sổ tiến độ cũ trên Drive là rác — xoá để lần chạy sau
        # không nhìn thấy một trạng thái mâu thuẫn (vừa xong vừa dở).
        rc('deletefile', f'{DRIVE}/{stem}.progress.json')
    print(f'  {"✅" if ok else "❌"} đẩy mảnh {k}{" (phần dở)" if partial_ok else ""} lên Drive')
    return ok

def launch(k, gpu):
    args = [sys.executable,'-u','scripts/encode_corpus.py','--config',CONFIG,
            '--shard',f'{k}/{N_SHARDS}','--device','cuda','--batch-size',str(BATCH_SIZE)]
    if k in partial:
        args.append('--resume')
    log = open(f'/kaggle/working/shard{k}.log','w')
    p = subprocess.Popen(args, stdout=log, stderr=subprocess.STDOUT,
                         env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)))
    print(f'▶ mảnh {k} trên GPU{gpu}{" (resume)" if k in partial else ""}')
    return {'k': k, 'p': p, 'log': log}

if not FORCE_MERGE_ONLY:
    queue, slots, hit_deadline = list(mine), {g: None for g in range(max(N_GPU, 1))}, False
    while queue or any(slots.values()):
        for gpu, job in slots.items():
            if job is None and queue and time.time() < DEADLINE:
                slots[gpu] = launch(queue.pop(0), gpu)
        time.sleep(30)
        for gpu, job in list(slots.items()):
            if job is None:
                continue
            rcode = job['p'].poll()
            if rcode is None:
                tail = subprocess.run(['tail','-1',f"/kaggle/working/shard{job['k']}.log"],
                                      capture_output=True,text=True).stdout.strip()
                print(f"[mảnh {job['k']}] {tail}")
                continue
            job['log'].close()
            if rcode == 0:
                push(job['k'])
            elif rcode == 130:
                push(job['k'], partial_ok=True)   # bị SIGINT vì chạm hạn
            else:
                raise RuntimeError(f"mảnh {job['k']} lỗi (mã {rcode}) — xem /kaggle/working/shard{job['k']}.log")
            slots[gpu] = None
        if time.time() >= DEADLINE and any(slots.values()) and not hit_deadline:
            hit_deadline = True
            print(f'⏰ chạm hạn {DEADLINE_HOURS}h — dừng có trật tự, giữ lại phần đã encode')
            for job in slots.values():
                if job:
                    job['p'].send_signal(signal.SIGINT)
            queue = []
    print(f'\nxong phiên này sau {(time.time()-T0)/3600:.2f}h')

## 7. Ghép — tự động khi đủ N mảnh

Ai xong sau cùng thì người đó ghép. Khoá `merging.lock` trên Drive để hai tài khoản không ghép
cùng lúc — **best-effort**, vì Drive không có thao tác kiểm-và-đặt nguyên tử. Trường hợp xấu nhất
là ghép thừa một lần, không phải ghép sai: `merge_embeddings.py` kiểm tính nhất quán trước khi ghi.

In [ ]:
done, partial = drive_state()
print(f'trên Drive: {len(done)}/{N_SHARDS} mảnh hoàn tất · {sorted(done)}')

if DO_MERGE and len(done) == N_SHARDS:
    lock = json.loads(rc('lsjson', DRIVE).stdout or '[]')
    if any(f['Name'] == 'merging.lock' for f in lock) and not FORCE_MERGE_ONLY:
        print('⏭ tài khoản khác đang ghép (có merging.lock). Bỏ qua.')
    else:
        open('/kaggle/working/merging.lock','w').write(json.dumps({'t': time.time()}))
        rc('copy','/kaggle/working', DRIVE,'--include=merging.lock', check=True)
        os.makedirs(f'{WORK}/data/shards', exist_ok=True)
        rc('copy', DRIVE, f'{WORK}/data/shards','-P','--stats=60s',
           '--include=embeddings.shard*.npy','--include=embeddings.shard*.meta.json',
           '--exclude=final/**', check=True)
        r = subprocess.run([sys.executable,'scripts/merge_embeddings.py','--shards','data/shards',
                            '--out','data/embeddings.npy','--dtype','float16'],
                           capture_output=True,text=True)
        print(r.stdout[-2500:] or r.stderr[-2500:])
        if r.returncode == 0:
            rc('copy', f'{WORK}/data', f'{DRIVE}/final','-P',
               '--include=embeddings.npy','--include=embeddings.meta.json', check=True)
            print('\n✅ BẢN CUỐI:', f'{DRIVE}/final')
            print(rc('lsl', f'{DRIVE}/final').stdout)
        rc('deletefile', f'{DRIVE}/merging.lock')
        assert r.returncode == 0, 'ghép thất bại — đọc thông báo ở trên, ĐỪNG dùng file này'
elif DO_MERGE:
    print(f'⏳ còn thiếu mảnh {sorted(set(range(N_SHARDS)) - done)} — chạy lại notebook này',
          'ở bất kỳ tài khoản nào sau khi các mảnh kia xong, nó sẽ tự ghép.')

## 8. Tổng kết phiên

In [ ]:
done, partial = drive_state()
print(f'thời gian phiên : {(time.time()-T0)/3600:.2f}h')
print(f'hoàn tất        : {len(done)}/{N_SHARDS} · {sorted(done)}')
print(f'còn dở          : {sorted(partial)}')
print(f'còn thiếu hẳn   : {sorted(set(range(N_SHARDS)) - done - partial)}')
print()
print(rc('lsl', DRIVE).stdout)
if len(done) < N_SHARDS:
    print('→ Bấm Save & Run All lần nữa (ở tài khoản còn quota). Nó tự bỏ qua mảnh xong,',
          'resume mảnh dở, và ghép khi đủ.')